# Big Data — Gastos eleitorais nas eleições municipais do Nordeste

Este notebook é **separado** do estudo de Presidente e Governador. Mantém a mesma metodologia e organização do projeto principal, mas analisa somente **Prefeito e Vereador** nos nove estados do Nordeste.

**Anos:** 2016, 2020 e 2024. São os três pleitos municipais completos mais recentes. O ano de 2026 não é municipal.

**Pergunta central:** candidatos que declararam maiores despesas de campanha tiveram maior frequência de eleição?

**Métrica:** soma das despesas contratadas (`VR_DESPESA_CONTRATADA`) por `SQ_CANDIDATO`; se o arquivo não disponibilizar essa coluna, o código aceita despesas pagas como alternativa e informa o arquivo utilizado.

Fonte: Portal de Dados Abertos do TSE. Os dados declarados não representam necessariamente o custo total real da campanha.

## 1. Bibliotecas e configurações

In [ ]:
!pip -q install pandas requests openpyxl

import io, re, time, zipfile, unicodedata
import pandas as pd
import requests

ANOS = [2016, 2020, 2024]
ESTADOS_NE = ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"]
CARGOS = ["PREFEITO", "VEREADOR"]
API_TSE = "https://dadosabertos.tse.jus.br/api/3/action/package_show"
print("Configurações carregadas.")

## 2. Funções para localizar e baixar os arquivos oficiais do TSE

In [ ]:
import io
import re
import time
import zipfile
import unicodedata
import pandas as pd
import requests

ANOS = [2016, 2020, 2024]
ESTADOS_NE = ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"]
CARGOS = ["PREFEITO", "VEREADOR"]
API_TSE = "https://dadosabertos.tse.jus.br/api/3/action/package_show"

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    return re.sub(r"[^a-z0-9]+", " ", texto).strip()

def consultar_conjunto(slug):
    r = requests.get(API_TSE, params={"id": slug}, timeout=45)
    r.raise_for_status()
    j = r.json()
    return j["result"].get("resources", []) if j.get("success") else []

def escolher_recurso(recursos, termos):
    for recurso in recursos or []:
        nome = normalizar(recurso.get("name", ""))
        if all(t in nome for t in termos):
            return recurso
    return None

def localizar_recursos(ano):
    candidatos = consultar_conjunto(f"candidatos-{ano}")
    resultados = consultar_conjunto(f"resultados-{ano}")
    if not resultados:
        resultados = consultar_conjunto(f"resultados-eleitorais-{ano}")
    rc = (escolher_recurso(candidatos, ["consulta", "cand", "brasil"])
          or escolher_recurso(candidatos, ["consulta", "cand"])
          or escolher_recurso(candidatos, ["cand"]))
    rr = (escolher_recurso(resultados, ["votacao", "candidato", "brasil"])
          or escolher_recurso(resultados, ["votacao", "candidato"])
          or escolher_recurso(resultados, ["votacao", "nominal"]))
    url_despesas = (
        "https://cdn.tse.jus.br/estatistica/sead/odsele/prestacao_contas/"
        f"prestacao_de_contas_eleitorais_candidatos_{ano}.zip"
    )
    rd = {"name": f"prestacao_de_contas_eleitorais_candidatos_{ano}.zip",
          "url": url_despesas}
    faltantes = [nome for nome, recurso in
                 [("candidatos", rc), ("resultados", rr), ("despesas", rd)]
                 if recurso is None]
    if faltantes:
        raise ValueError(
            f"Recursos não encontrados em {ano}: {faltantes}. "
            f"Candidatos disponíveis: {[x.get('name') for x in candidatos or []][:15]}; "
            f"resultados disponíveis: {[x.get('name') for x in resultados or []][:15]}"
        )
    return {"candidatos": rc, "resultados": rr, "despesas": rd}

def baixar_conteudo(recurso, tentativas=3):
    erro = None
    for tentativa in range(1, tentativas + 1):
        try:
            print("Baixando:", recurso.get("name"))
            resp = requests.get(recurso["url"], timeout=300)
            resp.raise_for_status()
            return resp.content
        except Exception as exc:
            erro = exc
            print(f"Tentativa {tentativa}/{tentativas}: {exc}")
            if tentativa < tentativas:
                time.sleep(5)
    raise ConnectionError(f"Falha no download: {erro}")

def escolher_csv(arquivo_zip, tipo):
    nomes = [n for n in arquivo_zip.namelist()
             if n.lower().endswith((".csv", ".txt"))]
    norm = {n: normalizar(n) for n in nomes}
    if tipo == "candidatos":
        opcoes = [n for n in nomes if "consulta" in norm[n] and "cand" in norm[n]]
        if not opcoes:
            opcoes = [n for n in nomes if "cand" in norm[n] and "despesa" not in norm[n]]
    elif tipo == "despesas":
        opcoes = [n for n in nomes if "despesas contratadas" in norm[n]
                  and "receitas" not in norm[n]]
        if not opcoes:
            opcoes = [n for n in nomes if "despesas pagas" in norm[n]
                      and "receitas" not in norm[n]]
    elif tipo == "resultados":
        opcoes = [n for n in nomes if "votacao candidato" in norm[n]
                  or "votacao nominal" in norm[n]]
    else:
        opcoes = nomes
    if not opcoes:
        raise ValueError(f"CSV do tipo {tipo} não encontrado. Amostra: {nomes[:30]}")
    brasil = [n for n in opcoes if "brasil" in norm[n] or re.search(r"(^| )br( |$)", norm[n])]
    opcoes = brasil or opcoes
    opcoes.sort(key=len)
    return opcoes[0]

def baixar_e_ler(recurso, tipo):
    conteudo = baixar_conteudo(recurso)
    if zipfile.is_zipfile(io.BytesIO(conteudo)):
        with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
            nome = escolher_csv(z, tipo)
            print("CSV escolhido:", nome)
            with z.open(nome) as f:
                return pd.read_csv(f, sep=";", encoding="latin1", low_memory=False)
    return pd.read_csv(io.BytesIO(conteudo), sep=";", encoding="latin1", low_memory=False)

def encontrar_coluna(df, alternativas, obrigatoria=True):
    for nome in alternativas:
        if nome in df.columns:
            return nome
    if obrigatoria:
        raise ValueError(
            f"Não encontrei nenhuma destas colunas: {alternativas}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None

def converter_moeda(serie):
    texto = serie.astype(str).str.strip().str.replace("R$", "", regex=False)
    texto = texto.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(texto, errors="coerce").fillna(0)

def processar_despesas(recurso, ids_candidatos, ano):
    conteudo = baixar_conteudo(recurso)
    ids = set(map(str, ids_candidatos))
    totais = {}
    categorias = []

    def processar_csv(stream, nome):
        cab = pd.read_csv(stream, sep=";", encoding="latin1", nrows=0)
        stream.seek(0)
        col_id = encontrar_coluna(cab, ["SQ_CANDIDATO"])
        col_valor = encontrar_coluna(
            cab, ["VR_DESPESA_CONTRATADA", "VR_DESPESA_PAGA", "VR_DESPESA"]
        )
        col_categoria = encontrar_coluna(
            cab, ["DS_TIPO_DESPESA", "DS_DESPESA", "DS_ORIGEM_DESPESA"],
            obrigatoria=False
        )
        for chunk in pd.read_csv(stream, sep=";", encoding="latin1",
                                 chunksize=200_000, low_memory=False):
            chunk[col_id] = chunk[col_id].astype(str).str.strip()
            chunk = chunk[chunk[col_id].isin(ids)].copy()
            if chunk.empty:
                continue
            chunk[col_valor] = converter_moeda(chunk[col_valor])
            if col_categoria is None:
                chunk["CATEGORIA_AUX"] = "Categoria não informada"
                categoria = "CATEGORIA_AUX"
            else:
                categoria = col_categoria
                chunk[categoria] = chunk[categoria].fillna(
                    "Categoria não informada"
                ).astype(str)
            soma = chunk.groupby(col_id)[col_valor].sum()
            for cid, valor in soma.items():
                totais[str(cid)] = totais.get(str(cid), 0.0) + float(valor)
            grupo = chunk.groupby([col_id, categoria])[col_valor].sum().reset_index()
            grupo.columns = ["SQ_CANDIDATO", "Categoria de despesa", "Valor gasto (R$)"]
            grupo["Ano"] = ano
            categorias.append(grupo)

    if zipfile.is_zipfile(io.BytesIO(conteudo)):
        with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
            nomes = [n for n in z.namelist()
                     if n.lower().endswith((".csv", ".txt"))
                     and "despesa" in normalizar(n)
                     and "receita" not in normalizar(n)]
            br = [n for n in nomes if "brasil" in normalizar(n)
                  or re.search(r"(^| )br( |$)", normalizar(n))]
            if br:
                nomes = br
            contratadas = [n for n in nomes if "despesas contratadas" in normalizar(n)]
            pagas = [n for n in nomes if "despesas pagas" in normalizar(n)]
            nomes = contratadas or pagas
            if not nomes:
                raise ValueError("Não encontrei arquivo de despesas contratadas/pagas no ZIP.")
            for nome in nomes:
                with z.open(nome) as f:
                    processar_csv(io.BytesIO(f.read()), nome)
                print("Arquivo de despesas utilizado:", nome)
    else:
        processar_csv(io.BytesIO(conteudo), recurso.get("name", ""))
    cat_df = (pd.concat(categorias, ignore_index=True) if categorias else
              pd.DataFrame(columns=["SQ_CANDIDATO", "Categoria de despesa",
                                    "Valor gasto (R$)", "Ano"]))
    return totais, cat_df

def identificar_eleitos(recurso, ids_candidatos):
    conteudo = baixar_conteudo(recurso)
    ids = set(map(str, ids_candidatos))
    eleitos = set()
    arquivos = []
    if zipfile.is_zipfile(io.BytesIO(conteudo)):
        with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
            nomes = [n for n in z.namelist()
                     if n.lower().endswith((".csv", ".txt"))
                     and ("votacao_candidato" in n.lower()
                          or "votacao_nominal" in n.lower())]
            br = [n for n in nomes if "brasil" in n.lower() or "_br" in n.lower()]
            arquivos = br or nomes
            if not arquivos:
                raise ValueError("Não encontrei arquivo de votação nominal no ZIP.")
            for nome in arquivos:
                with z.open(nome) as f:
                    for parte in pd.read_csv(f, sep=";", encoding="latin1",
                                             chunksize=200_000, low_memory=False):
                        col_id = encontrar_coluna(parte, ["SQ_CANDIDATO"])
                        col_sit = encontrar_coluna(
                            parte, ["DS_SIT_TOT_TURNO", "DS_SITUACAO_TSE"]
                        )
                        parte[col_id] = parte[col_id].astype(str).str.strip()
                        parte = parte[parte[col_id].isin(ids)]
                        situacao = parte[col_sit].fillna("").astype(str).str.upper()
                        filtro = (situacao.str.contains("ELEITO")
                                  & ~situacao.str.contains("NAO ELEITO")
                                  & ~situacao.str.contains("NÃO ELEITO"))
                        eleitos.update(parte.loc[filtro, col_id].unique())
    else:
        for parte in pd.read_csv(io.BytesIO(conteudo), sep=";", encoding="latin1",
                                 chunksize=200_000, low_memory=False):
            col_id = encontrar_coluna(parte, ["SQ_CANDIDATO"])
            col_sit = encontrar_coluna(parte, ["DS_SIT_TOT_TURNO", "DS_SITUACAO_TSE"])
            parte[col_id] = parte[col_id].astype(str).str.strip()
            parte = parte[parte[col_id].isin(ids)]
            situacao = parte[col_sit].fillna("").astype(str).str.upper()
            filtro = (situacao.str.contains("ELEITO")
                      & ~situacao.str.contains("NAO ELEITO")
                      & ~situacao.str.contains("NÃO ELEITO"))
            eleitos.update(parte.loc[filtro, col_id].unique())
    return eleitos


## 3. Processar candidatos, despesas e situação eleitoral

O filtro considera os nove estados e os cargos de Prefeito e Vereador. A identificação de eleito é feita pela situação totalizada do candidato no arquivo de votação do TSE.

In [ ]:
registros = []
categorias_por_ano = {}
erros = []

for ano in ANOS:
    print("\n" + "=" * 70)
    print("PROCESSANDO ELEIÇÕES MUNICIPAIS:", ano)
    try:
        recursos = localizar_recursos(ano)
        candidatos = baixar_e_ler(recursos["candidatos"], "candidatos")

        col_id = encontrar_coluna(candidatos, ["SQ_CANDIDATO"])
        col_nome = encontrar_coluna(candidatos, ["NM_URNA_CANDIDATO", "NM_CANDIDATO"])
        col_partido = encontrar_coluna(candidatos, ["SG_PARTIDO"])
        col_cargo = encontrar_coluna(candidatos, ["DS_CARGO"])
        col_uf = encontrar_coluna(candidatos, ["SG_UF"])
        col_municipio = encontrar_coluna(
            candidatos, ["NM_UE", "NM_MUNICIPIO", "NM_MUNICIPIO_NASCIMENTO"],
            obrigatoria=False
        )

        candidatos[col_id] = candidatos[col_id].astype(str).str.strip()
        filtro = (
            candidatos[col_uf].isin(ESTADOS_NE)
            & candidatos[col_cargo].astype(str).str.upper().str.strip().isin(CARGOS)
        )
        selecionados = candidatos.loc[filtro].copy()

        if selecionados.empty:
            raise ValueError("Nenhum candidato a Prefeito/Vereador encontrado nos estados do Nordeste.")

        ids = set(selecionados[col_id].dropna().astype(str))
        despesas, categorias = processar_despesas(recursos["despesas"], ids, ano)
        categorias_por_ano[ano] = categorias
        eleitos = identificar_eleitos(recursos["resultados"], ids)

        for _, cand in selecionados.iterrows():
            cid = str(cand[col_id])
            municipio = cand[col_municipio] if col_municipio else "Não informado"
            registros.append({
                "Ano": ano,
                "Cargo": str(cand[col_cargo]).upper().strip(),
                "Estado": cand[col_uf],
                "Município": municipio,
                "SQ_CANDIDATO": cid,
                "Candidato": cand[col_nome],
                "Partido": cand[col_partido],
                "Gasto total (R$)": despesas.get(cid, 0.0),
                "Eleito": "Sim" if cid in eleitos else "Não"
            })

        print("Candidatos selecionados:", len(selecionados))
        print("Candidatos com despesas:", sum(despesas.get(cid, 0) > 0 for cid in ids))
        print("Candidatos identificados como eleitos:", len(eleitos))
        print("Ano processado.")
    except Exception as erro:
        erros.append({"Ano": ano, "Erro": f"{type(erro).__name__}: {erro}"})
        print("ERRO:", erros[-1]["Erro"])

print("\nResumo de erros:")
display(pd.DataFrame(erros))

## 4. Tabela geral de candidatos

In [ ]:
colunas = [
    "Ano", "Cargo", "Estado", "Município", "SQ_CANDIDATO",
    "Candidato", "Partido", "Gasto total (R$)", "Eleito"
]
tabela_geral = pd.DataFrame(registros, columns=colunas)

if tabela_geral.empty:
    print("Nenhum ano foi processado. Verifique a tabela de erros acima.")
else:
    tabela_geral["Gasto total (R$)"] = pd.to_numeric(
        tabela_geral["Gasto total (R$)"], errors="coerce"
    ).fillna(0)
    tabela_geral = tabela_geral.drop_duplicates(
        subset=["Ano", "SQ_CANDIDATO"], keep="first"
    ).sort_values(
        ["Ano", "Estado", "Município", "Cargo", "Gasto total (R$)"],
        ascending=[True, True, True, True, False]
    ).reset_index(drop=True)
    display(tabela_geral.style.format({"Gasto total (R$)": "R$ {:,.2f}"}))
    print("Total de registros:", len(tabela_geral))

## 5. Os 10 candidatos que mais gastaram por ano e cargo

In [ ]:
if tabela_geral.empty:
    top10 = pd.DataFrame(columns=tabela_geral.columns)
else:
    top10 = (
        tabela_geral.sort_values("Gasto total (R$)", ascending=False)
        .groupby(["Ano", "Cargo"], group_keys=False)
        .head(10).reset_index(drop=True)
    )
display(top10.style.format({"Gasto total (R$)": "R$ {:,.2f}"}))

## 6. Comparar gastos e eleição

A comparação abaixo calcula, por ano e cargo, quantos candidatos foram eleitos e a proporção de eleitos entre os candidatos com e sem despesas registradas. Isso descreve uma associação; não prova que gastar mais causou a eleição.

In [ ]:
if tabela_geral.empty:
    resumo_eleicao = pd.DataFrame()
else:
    resumo_eleicao = (
        tabela_geral.groupby(["Ano", "Cargo"], as_index=False)
        .agg(
            Candidatos=("SQ_CANDIDATO", "nunique"),
            Eleitos=("Eleito", lambda s: (s == "Sim").sum()),
            Gasto_medio=("Gasto total (R$)", "mean"),
            Gasto_mediano=("Gasto total (R$)", "median"),
            Gasto_total=("Gasto total (R$)", "sum")
        )
    )
    resumo_eleicao["Percentual_eleitos"] = (
        resumo_eleicao["Eleitos"] / resumo_eleicao["Candidatos"] * 100
    )
    display(resumo_eleicao.style.format({
        "Gasto_medio": "R$ {:,.2f}", "Gasto_mediano": "R$ {:,.2f}",
        "Gasto_total": "R$ {:,.2f}", "Percentual_eleitos": "{:.2f}%"
    }))

# Faixas de gastos para observar a proporção de eleitos em cada faixa.
if not tabela_geral.empty:
    base_faixas = tabela_geral.copy()
    base_faixas["Faixa de gastos"] = base_faixas.groupby(
        ["Ano", "Cargo"]
    )["Gasto total (R$)"].transform(
        lambda s: pd.qcut(s.rank(method="first"), q=4,
                          labels=["Menor gasto", "2º grupo", "3º grupo", "Maior gasto"])
    )
    comparacao_faixas = (
        base_faixas.groupby(["Ano", "Cargo", "Faixa de gastos"], observed=False)
        .agg(
            Candidatos=("SQ_CANDIDATO", "nunique"),
            Eleitos=("Eleito", lambda s: (s == "Sim").sum()),
            Gasto_medio=("Gasto total (R$)", "mean")
        ).reset_index()
    )
    comparacao_faixas["Percentual_eleitos"] = (
        comparacao_faixas["Eleitos"] / comparacao_faixas["Candidatos"] * 100
    )
    display(comparacao_faixas.style.format({
        "Gasto_medio": "R$ {:,.2f}", "Percentual_eleitos": "{:.2f}%"
    }))

## 7. Gastos por estado e município

In [ ]:
if not tabela_geral.empty:
    gastos_estado = (
        tabela_geral.groupby(["Ano", "Estado", "Cargo"], as_index=False)
        .agg(
            Candidatos=("SQ_CANDIDATO", "nunique"),
            Gasto_total=("Gasto total (R$)", "sum"),
            Gasto_medio=("Gasto total (R$)", "mean"),
            Eleitos=("Eleito", lambda s: (s == "Sim").sum())
        )
        .sort_values(["Ano", "Cargo", "Gasto_total"], ascending=[True, True, False])
    )
    display(gastos_estado.style.format({
        "Gasto_total": "R$ {:,.2f}", "Gasto_medio": "R$ {:,.2f}"
    }))

    gastos_municipio = (
        tabela_geral.groupby(["Ano", "Estado", "Município", "Cargo"], as_index=False)
        .agg(
            Candidatos=("SQ_CANDIDATO", "nunique"),
            Gasto_total=("Gasto total (R$)", "sum"),
            Gasto_medio=("Gasto total (R$)", "mean"),
            Eleitos=("Eleito", lambda s: (s == "Sim").sum())
        )
        .sort_values(["Ano", "Estado", "Gasto_total"], ascending=[True, True, False])
    )
    display(gastos_municipio.head(100).style.format({
        "Gasto_total": "R$ {:,.2f}", "Gasto_medio": "R$ {:,.2f}"
    }))

## 8. Categorias de despesas dos 10 maiores gastadores

In [ ]:
blocos = []
if not top10.empty:
    for ano in sorted(top10["Ano"].unique()):
        cat = categorias_por_ano.get(int(ano), pd.DataFrame())
        if cat.empty:
            continue
        ids_top = set(top10.loc[top10["Ano"] == ano, "SQ_CANDIDATO"].astype(str))
        cat = cat[cat["SQ_CANDIDATO"].astype(str).isin(ids_top)].copy()
        nomes = top10.loc[top10["Ano"] == ano, ["SQ_CANDIDATO", "Candidato"]].drop_duplicates()
        cat = cat.merge(nomes, on="SQ_CANDIDATO", how="left")
        blocos.append(cat[["Ano", "Candidato", "Categoria de despesa", "Valor gasto (R$)"]])

gastos_por_categoria = (
    pd.concat(blocos, ignore_index=True).groupby(
        ["Ano", "Candidato", "Categoria de despesa"], as_index=False
    )["Valor gasto (R$)"].sum().sort_values(
        ["Ano", "Candidato", "Valor gasto (R$)"], ascending=[True, True, False]
    ) if blocos else
    pd.DataFrame(columns=["Ano", "Candidato", "Categoria de despesa", "Valor gasto (R$)"])
)
display(gastos_por_categoria.style.format({"Valor gasto (R$)": "R$ {:,.2f}"}))

## 9. Exportar tabelas para Excel

In [ ]:
arquivo_excel = "analise_gastos_municipais_nordeste.xlsx"
with pd.ExcelWriter(arquivo_excel) as writer:
    tabela_geral.to_excel(writer, sheet_name="Todos candidatos", index=False)
    top10.to_excel(writer, sheet_name="Top 10 gastos", index=False)
    resumo_eleicao.to_excel(writer, sheet_name="Resumo eleicao", index=False)
    comparacao_faixas.to_excel(writer, sheet_name="Faixas de gastos", index=False)
    gastos_estado.to_excel(writer, sheet_name="Gastos por estado", index=False)
    gastos_municipio.to_excel(writer, sheet_name="Gastos por municipio", index=False)
    gastos_por_categoria.to_excel(writer, sheet_name="Categorias", index=False)
    pd.DataFrame(erros).to_excel(writer, sheet_name="Erros", index=False)

print("Planilha criada:", arquivo_excel)
try:
    from google.colab import files
    files.download(arquivo_excel)
except ImportError:
    print("Arquivo salvo na pasta atual.")

## 10. Conferências antes de apresentar

- Verifique se os três anos foram processados sem erros.
- Confira se o TSE disponibilizou os arquivos consolidados de votação e despesas para cada ano.
- O campo `Eleito` depende da situação totalizada no arquivo de votação; candidaturas sub judice, anuladas ou com atualização posterior podem exigir conferência.
- A tabela Top 10 separa Prefeito e Vereador, mas reúne os nove estados. Use os filtros do Power BI/Excel para detalhar por estado e município.
- Compare candidatos do mesmo ano e cargo. Não compare diretamente o gasto nominal de 2016 com 2024 sem considerar inflação.
- Despesa declarada não significa que esse valor, isoladamente, determinou a eleição. O estudo identifica relações, não causalidade.

**Observação técnica:** a base de candidaturas costuma trazer o município na coluna `NM_UE`. Se o arquivo de determinado ano usar outro nome, a célula de seleção de colunas informa quais campos foram encontrados; ajuste a lista de alternativas conforme o cabeçalho oficial daquele ano.